In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import matplotlib.pyplot as plt

In [13]:
from torch.utils.data import Dataset, DataLoader
# from skimage import io, transform
# import os

# transform = torchvision.transforms.Compose([
#     torchvision.transforms.ToTensor(),
# ])

# class FaceLandmarksDataset(Dataset):
#     """Face Landmarks dataset."""

#     def __init__(self, csv_file, img_root_dir, transform=None):
#         self.landmarks_frame = []
#         file = open(csv_file, "r")
#         while True:
#             content=file.readline()
#             if not content:
#                 break
#             self.landmarks_frame.append(content.split(","))
#         file.close()
#         self.landmarks_frame = self.landmarks_frame[2:]

#         self.img_root_dir = img_root_dir
#         self.transform = transform

#     def __len__(self):
#         return len(self.landmarks_frame)

#     def __getitem__(self, idx):
#         if torch.is_tensor(idx):
#             idx = idx.tolist()
#         img_name = os.path.join(self.img_root_dir, self.landmarks_frame[idx][0])
#         image = io.imread(img_name)
#         landmarks = self.landmarks_frame[idx][1:]
#         landmarks = np.array([landmarks], dtype=float).reshape(-1, 2)
#         scaled_landmarks = landmarks.copy()
#         scaled_landmarks[:, 0], scaled_landmarks[:, 1] = scaled_landmarks[:, 0] / 178, scaled_landmarks[:, 1] / 218
#         if self.transform:
#             image_tr = self.transform(image)
#         scaled_landmarks = scaled_landmarks.reshape(-1)
#         sample = {'image': image, 'image_tr':image_tr, 'landmarks': landmarks, 'scaled_landmarks': scaled_landmarks}

#         return sample


In [21]:
from torchvision import datasets
from torchvision import transforms
full_dataset = datasets.CelebA(root='./',split="train", target_type="landmarks" ,download=True, transform=torchvision.transforms.Compose([transforms.PILToTensor()]))

In [22]:
# full_dataset = FaceLandmarksDataset("list_landmarks_align_celeba.csv", "./img_align_celeba/img_align_celeba", transform=transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = torch.utils.data.random_split(full_dataset, [train_size, test_size])

print(train_set.__len__(), test_set.__len__())

130216 32554


In [23]:
# sample = train_set[1000]
# landmarks = sample['landmarks']
# image = sample['image']
# print(sample['image_tr'][0,:5,0:5])
# print(sample['scaled_landmarks'].shape)
# def show_landmarks(image, landmarks):
#     """Show image with landmarks"""
#     plt.imshow(image)
#     plt.scatter(landmarks[:, 0], landmarks[:, 1], s=5, marker='.', c='r')
#     plt.pause(0.001)  # pause a bit so that plots are updated

print(train_set[0])


(tensor([[[126, 130, 132,  ..., 130, 135, 133],
         [126, 130, 132,  ..., 130, 135, 133],
         [126, 130, 132,  ..., 131, 135, 133],
         ...,
         [186, 185, 186,  ..., 223, 228, 228],
         [192, 190, 189,  ..., 223, 226, 226],
         [192, 190, 189,  ..., 223, 226, 226]],

        [[ 38,  44,  53,  ...,  85,  95,  95],
         [ 38,  44,  53,  ...,  85,  95,  95],
         [ 38,  44,  53,  ...,  86,  95,  95],
         ...,
         [113, 112, 111,  ..., 161, 165, 165],
         [118, 116, 114,  ..., 161, 162, 162],
         [118, 116, 114,  ..., 161, 162, 162]],

        [[ 37,  43,  49,  ...,  92, 106, 108],
         [ 37,  43,  49,  ...,  92, 106, 108],
         [ 37,  43,  49,  ...,  93, 106, 108],
         ...,
         [ 72,  71,  71,  ..., 120, 124, 124],
         [ 79,  77,  75,  ..., 120, 124, 124],
         [ 79,  77,  75,  ..., 120, 124, 124]]], dtype=torch.uint8), tensor([ 69, 111, 108, 112,  89, 133,  75, 153, 103, 151]))


In [24]:
dataloader = DataLoader(train_set, batch_size=4,
                        shuffle=True, num_workers=0)

In [25]:
for i in dataloader:
    print(i[0].shape)
    landmarks = i[1]
    image = i[0]
    print(image, landmarks)
    break
    # print(i['scaled_landmarks'].shape, i['scaled_landmarks'])
    # plt.figure()
    # show_landmarks(image,
    #             landmarks)
    # plt.show()
    # break

torch.Size([4, 3, 218, 178])
tensor([[[[ 31,  30,  32,  ...,  40,  36,  37],
          [ 31,  30,  32,  ...,  38,  38,  38],
          [ 30,  30,  32,  ...,  37,  36,  36],
          ...,
          [ 40,  40,  41,  ..., 171, 199, 211],
          [ 40,  45,  40,  ..., 168, 204, 207],
          [ 40,  45,  40,  ..., 164, 189, 194]],

         [[ 37,  36,  38,  ...,  40,  34,  34],
          [ 37,  36,  38,  ...,  37,  36,  36],
          [ 36,  36,  38,  ...,  39,  36,  36],
          ...,
          [ 30,  30,  31,  ..., 139, 167, 179],
          [ 30,  35,  30,  ..., 139, 175, 181],
          [ 30,  35,  30,  ..., 138, 163, 169]],

         [[ 53,  52,  54,  ...,  52,  47,  45],
          [ 53,  52,  54,  ...,  51,  49,  47],
          [ 52,  52,  54,  ...,  52,  48,  48],
          ...,
          [ 28,  28,  29,  ..., 142, 170, 182],
          [ 28,  33,  28,  ..., 141, 179, 184],
          [ 28,  33,  28,  ..., 139, 166, 172]]],


        [[[ 14,  20,  13,  ...,  65,  68,  68],
      

# Model

In [26]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_chanels, **kwargs):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_chanels, **kwargs)
        self.bn = nn.BatchNorm2d(out_chanels)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))

class InceptionBlock(nn.Module):
    def __init__(self,  in_channels,  out_1x1, red_3x3, out_3x3, red_5x5, out_5x5, out_pool):

        super(InceptionBlock, self).__init__()
        self.branch1 = ConvBlock(in_channels, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBlock(in_channels, red_3x3, kernel_size=1, padding=0),
            ConvBlock(red_3x3, out_3x3, kernel_size=3, padding=1))
        self.branch3 = nn.Sequential(
            ConvBlock(in_channels, red_5x5, kernel_size=1),
            ConvBlock(red_5x5, out_5x5, kernel_size=5, padding=2))
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, padding=1, stride=1),
            ConvBlock(in_channels, out_pool, kernel_size=1))

    def forward(self, x):
        branches = (self.branch1, self.branch2, self.branch3, self.branch4)
        return torch.cat([branch(x) for branch in branches], 1)

In [27]:
class InceptionModel(nn.Module):
    def __init__(self, aux = False, residual = True, num_classes = 1000):
        super(InceptionModel, self).__init__()
        self.aux = aux
        self.residual  = residual
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(512, num_classes)

        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.avgpool = nn.AvgPool2d(kernel_size=(7,6), stride=1)

        self.conv1 = ConvBlock(in_channels=3, out_chanels=64, kernel_size=(7,7), stride=(2,2), padding=(3,3))
        self.conv2 = ConvBlock(in_channels=64, out_chanels=192, kernel_size=3, stride=1, padding=1)

        self.incept3a = InceptionBlock(in_channels=192, out_1x1=64, red_3x3=96, out_3x3=128, red_5x5=16, out_5x5=32, out_pool=32)
        self.incept3b = InceptionBlock(in_channels=256, out_1x1=128, red_3x3=128, out_3x3=192, red_5x5=32, out_5x5=112, out_pool=80)

        self.incept4a = InceptionBlock(in_channels=512, out_1x1=192, red_3x3=96, out_3x3=208, red_5x5=16, out_5x5=48, out_pool=64)
        self.incept4b = InceptionBlock(in_channels=512, out_1x1=160, red_3x3=112, out_3x3=224, red_5x5=24, out_5x5=64, out_pool=64)

        self.incept5a = InceptionBlock(in_channels=512, out_1x1=256, red_3x3=160, out_3x3=320, red_5x5=32, out_5x5=128, out_pool=128)
        self.incept5b = InceptionBlock(in_channels=832, out_1x1=128, red_3x3=112, out_3x3=256, red_5x5=32, out_5x5=64, out_pool=64)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.incept3a(x)
        x = self.incept3b(x)
        residual = self.maxpool(x)

        x = self.incept4a(residual)
        x = self.incept4b(x)
        x += residual
        residual = self.maxpool(x)

        x = self.incept5a(residual)
        x = self.incept5b(x)
        x += residual
        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.dropout(x)
        x = F.relu(self.fc(x))

        return x

In [28]:
dummy_input = torch.rand(8,3,218,178)
print(dummy_input.shape)

torch.Size([8, 3, 218, 178])


In [29]:
net = InceptionModel(num_classes=10)
out = net(dummy_input)

print(out.shape)

torch.Size([8, 10])


In [30]:
net

InceptionModel(
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=512, out_features=10, bias=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (avgpool): AvgPool2d(kernel_size=(7, 6), stride=1, padding=0)
  (conv1): ConvBlock(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv2): ConvBlock(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (incept3a): InceptionBlock(
    (branch1): ConvBlock(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (branch2): Sequential(
      (0): ConvBlock(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
     

In [31]:
from torchvision import models
class Network(nn.Module):
    def __init__(self,num_classes=10):
        super().__init__()
        self.model=models.resnet18()
        self.model.conv1=nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.fc=nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        x=F.relu(self.model(x))
        return x

# Training

In [36]:
def inference(model, device):
    model.eval()
    with torch.no_grad():
        data = test_set[-1]
        image, target = torch.tensor(data["image_tr"], dtype=torch.float32).unsqueeze(0).to(device),  torch.tensor(data["scaled_landmarks"],dtype=torch.float32).unsqueeze(0).to(device)
        print(image.shape)
        output = model(image)
        landmarks = np.array([output.cpu().detach()], dtype=float).reshape(-1, 2)
        landmarks[:, 0], landmarks[:, 1] = landmarks[:, 0] * 178, landmarks[:, 1] * 218
        landmarks = np.int8(landmarks)
        loss = F.mse_loss(output, target, reduction='sum').item()
        print('loss: ', loss, 'out: ', landmarks)
        # plt.figure()
        # show_landmarks(data["image"], data["landmarks"])
        # plt.show()

        # plt.figure()
        # show_landmarks(data["image"], landmarks)
        # plt.show()

def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data) in enumerate(train_loader):
        image, target = data[0].to(device).float(), data[1].to(device).float()
        optimizer.zero_grad()
        output = model(image)
        loss = F.mse_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data[0]), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data in test_loader:
            image, target = data[0].to(device).float(), data[1].to(device).float()
            output = model(image)
            test_loss += F.mse_loss(output, target, reduction='sum').item()  # sum up batch loss
    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}\n'.format(test_loss))


def main():
    epochs = 1
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)
    train_loader = DataLoader(train_set, batch_size=64,
                        shuffle=True, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=64,
                        shuffle=True, num_workers=0)
    # model = InceptionModel(num_classes=10)
    model = Network(10)
    model.to(device)
    optimizer = torch.optim.Adadelta(model.parameters(), lr=0.001)
    scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
    for epoch in range(1, epochs + 1):
        train(model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()
    torch.save(model.state_dict(), "./own_res_incept5.pt")
    inference(model, device)

In [37]:
from torch.optim.lr_scheduler import StepLR
main()

Using device: cuda
Train Epoch: 1 [0/130216 (0%)]	Loss: 12941.817383
Train Epoch: 1 [6400/130216 (5%)]	Loss: 12688.781250
Train Epoch: 1 [12800/130216 (10%)]	Loss: 12450.201172
Train Epoch: 1 [19200/130216 (15%)]	Loss: 12099.842773
Train Epoch: 1 [25600/130216 (20%)]	Loss: 11721.187500
Train Epoch: 1 [32000/130216 (25%)]	Loss: 11280.672852
Train Epoch: 1 [38400/130216 (29%)]	Loss: 10759.250000
Train Epoch: 1 [44800/130216 (34%)]	Loss: 10206.223633
Train Epoch: 1 [51200/130216 (39%)]	Loss: 9641.757812
Train Epoch: 1 [57600/130216 (44%)]	Loss: 9109.884766
Train Epoch: 1 [64000/130216 (49%)]	Loss: 8614.531250
Train Epoch: 1 [70400/130216 (54%)]	Loss: 8105.043945
Train Epoch: 1 [76800/130216 (59%)]	Loss: 7606.266602
Train Epoch: 1 [83200/130216 (64%)]	Loss: 7094.274902
Train Epoch: 1 [89600/130216 (69%)]	Loss: 6669.964355
Train Epoch: 1 [96000/130216 (74%)]	Loss: 6205.957520
Train Epoch: 1 [102400/130216 (79%)]	Loss: 5804.860840
Train Epoch: 1 [108800/130216 (84%)]	Loss: 5377.709473
Train 

TypeError: tuple indices must be integers or slices, not str

In [ ]:
import time
network = Network(10)
network.cuda()

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(network.parameters(), lr=0.0001)
train_loader = DataLoader(train_set, batch_size=64,
                    shuffle=True, num_workers=0)
valid_loader = DataLoader(test_set, batch_size=64,
                    shuffle=True, num_workers=0)
loss_min = np.inf
num_epochs = 10

start_time = time.time()
for epoch in range(1,num_epochs+1):

    loss_train = 0
    loss_valid = 0
    running_loss = 0

    network.train()
    for step in range(1,len(train_loader)+1):

        data = next(iter(train_loader))
        images, landmarks = data["image_tr"].float(), data["landmarks"].float()
        images = images.cuda()
        landmarks = landmarks.view(landmarks.size(0),-1).cuda()

        predictions = network(images)

        # clear all the gradients before calculating them
        optimizer.zero_grad()

        # find the loss for the current step
        loss_train_step = criterion(predictions, landmarks)

        # calculate the gradients
        loss_train_step.backward()

        # update the parameters
        optimizer.step()

        loss_train += loss_train_step.item()
        running_loss = loss_train/step

        # print_overwrite(step, len(train_loader), running_loss, 'train')

    network.eval()
    with torch.no_grad():

        for step in range(1,len(valid_loader)+1):

            data = next(iter(train_loader))
            images, landmarks = data["image_tr"].float(), data["landmarks"].float()
            images = images.cuda()
            landmarks = landmarks.view(landmarks.size(0),-1).cuda()

            predictions = network(images)

            # find the loss for the current step
            loss_valid_step = criterion(predictions, landmarks)

            loss_valid += loss_valid_step.item()
            running_loss = loss_valid/step

            # print_overwrite(step, len(valid_loader), running_loss, 'valid')

    loss_train /= len(train_loader)
    loss_valid /= len(valid_loader)

    print('\n--------------------------------------------------')
    print('Epoch: {}  Train Loss: {:.4f}  Valid Loss: {:.4f}'.format(epoch, loss_train, loss_valid))
    print('--------------------------------------------------')

    if loss_valid < loss_min:
        loss_min = loss_valid
        torch.save(network.state_dict(), '/content/face_landmarks.pth')
        print("\nMinimum Validation Loss of {:.4f} at epoch {}/{}".format(loss_min, epoch, num_epochs))
        print('Model Saved\n')

print('Training Complete')
print("Total Elapsed Time : {} s".format(time.time()-start_time))